# Preproccessing of the dataset

In [142]:
# Model creation
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.models import load_model

from egg_hunt.config import RAW_DATA, CACHE, MODEL_PATH, CLASS_NAMES
from egg_hunt.preprocessing import load_split, get_cropped_arrays
from egg_hunt.model import build_model

SEED = 42

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# ---------------------------------------
# Loading the Data
# ---------------------------------------

In [143]:
train_df = load_split(RAW_DATA / "train/train_labels.json", RAW_DATA / "train/train_data")
test_df  = load_split(RAW_DATA / "test/test_labels.json",   RAW_DATA / "test/test_data")

# ---------------------------------------
# Model Creation
# ---------------------------------------

In [144]:
X_train, y_train, X_test, y_test = get_cropped_arrays(train_df, test_df, cache_path=CACHE)

# Stratified train/validation split (also shuffles – no extra shuffle needed)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED, stratify=y_train
)
del X_train, y_train          # free ~1.6 GB

pd.DataFrame({"train": y_tr.sum(axis=0).astype(int),
              "val":   y_val.sum(axis=0).astype(int)}, index=CLASS_NAMES)

,train,val
Ascaris lumbricoides,807,202
Capillaria philippinensis,800,200
Enterobius vermicularis,800,200
Fasciolopsis buski,800,200
Hookworm egg,808,202
Hymenolepis diminuta,800,200
Hymenolepis nana,801,200
Opisthorchis viverrine,800,200
Paragonimus spp,800,200
Taenia spp. egg,808,203


In [141]:
model = build_model(augment=False, seed=SEED)
model.summary()

Model: "egg_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ preprocessing (Sequential)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ cnn (Sequential)                │ (None, 28, 28, 128)    │        93,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Sequential)         │ (None, 11)             │    12,846,603 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 12,939,851 (49.36 MB)

 Trainable params: 12,939,851 (49.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
MODEL_PATH.parent.mkdir(exist_ok=True)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
        ModelCheckpoint(MODEL_PATH, monitor="val_loss", save_best_only=True),
    ],
)

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ["loss", "accuracy"]):
    ax.plot(history.history[metric], label="train")
    ax.plot(history.history[f"val_{metric}"], label="validation")
    ax.set_title(metric); ax.set_xlabel("epoch"); ax.legend()
plt.show()

In [ ]:
# Final evaluation on the test set
scores = model.evaluate(X_test, y_test, return_dict=True)
print({k: round(v, 4) for k, v in scores.items()})

y_true = y_test.argmax(axis=1)
y_pred = model.predict(X_test).argmax(axis=1)
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=CLASS_NAMES,
                                        xticks_rotation=90, normalize="true", values_format=".2f")
plt.show()